In [ ]:
# 2.5D preprocess on Kaggle -> HF. Each sampled position saved as 3 ADJACENT slices (prev,curr,next)
# stacked as channels, so RadImageNet gets real volumetric context. 12 positions/series, 384px.
import os, glob, time, numpy as np, pandas as pd, pydicom, cv2
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, create_repo
K=12; IMG=384; SHARD_STUDIES=30
HF_TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
api=HfApi(token=HF_TOKEN)
USER=api.whoami()['name']; REPO=f'{USER}/rsna-knee-384-25d'
print('HF user:', USER, '| repo:', REPO)
create_repo(REPO, repo_type='dataset', private=True, exist_ok=True, token=HF_TOKEN)
base=None
for r in sorted(glob.glob('/kaggle/input/*')):
    h=glob.glob(r+'/**/train_series.csv', recursive=True)
    if h: base=os.path.dirname(h[0]); break
print('base', base)
tr=pd.read_csv(os.path.join(base,'train.csv')); ts=pd.read_csv(os.path.join(base,'train_series.csv'))
studies=list(tr['StudyInstanceUID']); print('studies', len(studies))
PL={'Sagittal':0,'Coronal':1,'Axial':2}

In [ ]:
def win(a):
    lo,hi=np.percentile(a,0.5),np.percentile(a,99.5); a=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
    return (cv2.resize(a,(IMG,IMG),interpolation=cv2.INTER_AREA)*255).astype(np.uint8)
def read(p):
    d=pydicom.dcmread(p); a=d.pixel_array.astype(np.float32)
    return a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
def positions(n,k):
    if n<=0: return []
    if n<=k: return list(range(n))
    lo,hi=int(n*.12),int(n*.88)
    if hi<=lo: lo,hi=0,n
    return [int(round(x)) for x in np.linspace(lo,hi-1,k)]
def slc25(fs,idx):
    # 3 adjacent slices (prev,curr,next) -> (3,IMG,IMG) uint8; clamp at series ends
    n=len(fs); out=[]
    for j in (idx-1, idx, idx+1):
        j=max(0,min(n-1,j))
        try: out.append(win(read(fs[j])))
        except Exception: out.append(np.zeros((IMG,IMG),np.uint8))
    return np.stack(out)  # (3,IMG,IMG)
t0=time.time(); imgs=[]; meta=[]; sids=[]; shard=0
def flush():
    global imgs,meta,sids,shard
    if not imgs: return
    fn=f'shard_{shard:03d}.npz'
    np.savez_compressed(fn, img=np.stack(imgs), meta=np.array(meta), study_ids=np.array(sids))
    api.upload_file(path_or_fileobj=fn, path_in_repo=fn, repo_id=REPO, repo_type='dataset', token=HF_TOKEN)
    os.remove(fn)
    print(f'  pushed {fn} ({shard+1}) ~{(shard+1)*SHARD_STUDIES} studies {time.time()-t0:.0f}s', flush=True)
    imgs=[]; meta=[]; sids=[]; shard+=1
for si,study in enumerate(studies):
    for _,sr in ts[ts['StudyInstanceUID']==study].iterrows():
        sd=os.path.join(base,'train_series',study,sr['SeriesInstanceUID'])
        if not os.path.isdir(sd): continue
        fs=sorted(glob.glob(sd+'/*.dcm'))
        pid=PL.get(sr['Anatomical_Plane'],0); fb=int(sr['Fluid_Sensitive'])
        for idx in positions(len(fs),K):
            im=slc25(fs,idx)
            if im is not None: imgs.append(im); meta.append((si,pid,fb))
    sids.append(study)
    if (si+1)%SHARD_STUDIES==0: flush()
flush()
pd.DataFrame({'si':range(len(studies)),'StudyInstanceUID':studies}).to_csv('study_index.csv',index=False)
api.upload_file(path_or_fileobj='study_index.csv', path_in_repo='study_index.csv', repo_id=REPO, repo_type='dataset', token=HF_TOKEN)
print('DONE 2.5D -> HF', REPO, f'{time.time()-t0:.0f}s', 'shards', shard)